In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import col, current_timestamp
 
raw_df = spark.read.format("json").load(landing_path)
 
bronze_new_df = (
    raw_df
    .withColumn("ingest_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)
 
if spark.catalog.tableExists(bronze_table):
    existing_ids_df = spark.table(bronze_table).select("transaction_id")
    bronze_new_df = bronze_new_df.join(existing_ids_df, on="transaction_id", how="left_anti")
 
bronze_new_df.write.format("delta").mode("append").saveAsTable(bronze_table)
 
print(f"Inserted {bronze_new_df.count()} new bronze rows")
display(spark.table(bronze_table))